In [1]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

In [2]:
DATA_PATH = '../data'

meta = pd.read_csv(f'{DATA_PATH}/metadata.tsv', sep='\t')
paths = pd.read_csv(f'{DATA_PATH}/pathways_relative_abundance.tsv', sep='\t')
taxs = pd.read_csv(f'{DATA_PATH}/taxonomy_relative_abundance.tsv', sep='\t')

In [3]:
meta.head(5)

,study_name,sample_id,subject_id,body_site,antibiotics_current_use,study_condition,disease,age,age_category,gender,...,country,non_westernized,sequencing_platform,DNA_extraction_kit,PMID,number_reads,number_bases,minimum_read_length,median_read_length,NCBI_accession
0,VogtmannE_2016,MMRS11288076ST-27-0-0,MMRS11288076ST-27-0-0,stool,missing: not collected,control,healthy,65.0,adult,male,...,USA,no,IlluminaHiSeq,Gnome,27171425,67924141.0,5.793273e+09,1.0,89.0,ERR1293500;ERR1293499;ERR1293498;ERR1293497;ER...
1,VogtmannE_2016,MMRS11932626ST-27-0-0,MMRS11932626ST-27-0-0,stool,missing: not collected,control,healthy,65.0,adult,male,...,USA,no,IlluminaHiSeq,Gnome,27171425,44412536.0,3.583470e+09,5.0,85.0,ERR1293881;ERR1293880;ERR1293879;ERR1293878;ER...
2,VogtmannE_2016,MMRS12272136ST-27-0-0,MMRS12272136ST-27-0-0,stool,missing: not collected,CRC,CRC,51.0,adult,male,...,USA,no,IlluminaHiSeq,Gnome,27171425,67377964.0,5.546605e+09,5.0,84.0,ERR1293877;ERR1293876;ERR1293875;ERR1293874;ER...
3,VogtmannE_2016,MMRS14379078ST-27-0-0,MMRS14379078ST-27-0-0,stool,missing: not collected,CRC,CRC,53.0,adult,male,...,USA,no,IlluminaHiSeq,Gnome,27171425,65466210.0,5.623913e+09,2.0,88.0,ERR1293548;ERR1293547;ERR1293546;ERR1293545;ER...
4,VogtmannE_2016,MMRS14602194ST-27-0-0,MMRS14602194ST-27-0-0,stool,missing: not collected,CRC,CRC,84.0,senior,female,...,USA,no,IlluminaHiSeq,Gnome,27171425,79073471.0,6.862242e+09,45.0,90.0,ERR1293813;ERR1293812;ERR1293811;ERR1293810;ER...


In [4]:
paths.head(5)

,Unnamed: 0,CON-001,CON-002,CON-003,CON-004,CON-005,CON-006,CON-007,CON-008,CON-009,...,H1M513747,H1M213648,H1M513852,H1M512853,H1M512854,H1M512855,H2M414940,nHM511825,nHM612836,nHM412840
0,DTDPRHAMSYN-PWY:_dTDP-L-rhamnose_biosynthesis_I,2.936295,2.631166,2.408017,2.716068,1.926538,2.356993,2.487057,2.308210,3.651102,...,1.427225,2.049108,2.116564,1.805724,1.538030,2.008264,4.112660,1.957936,1.347956,1.568229
1,PWY-7219:_adenosine_ribonucleotides_de_novo_bi...,2.433814,2.638658,2.016642,2.609938,2.185638,1.815148,1.636217,2.114412,2.098454,...,1.439908,1.356308,2.364435,1.433533,1.602553,2.211219,2.020161,1.927275,1.362865,1.810064
2,PWY-6609:_adenine_and_adenosine_salvage_III,2.316291,2.495638,1.853177,2.503757,2.150012,1.778849,1.642103,1.961368,1.936527,...,1.457638,1.344844,1.624785,1.260803,1.471688,0.000000,1.737105,2.181463,0.000000,1.494750
3,PWY-5686:_UMP_biosynthesis,2.310229,2.607788,1.818348,2.630974,2.072914,1.907910,1.603411,1.944407,2.144515,...,1.864858,1.327595,3.486686,1.793705,1.617443,2.973900,2.704087,1.921157,1.916884,2.008608
4,TRNA-CHARGING-PWY:_tRNA_charging,2.284019,2.271212,1.787862,2.310204,2.049201,1.694257,1.574177,1.817008,1.742076,...,1.532837,1.357127,1.684153,0.000000,1.739814,0.000000,0.000000,1.661008,1.499639,1.833352


In [5]:
taxs.head(5)

,Species,SZAXPI003410-3,SZAXPI003411-6,SZAXPI003412-1,SZAXPI003413-4,SZAXPI003414-3,SZAXPI003415-12,SZAXPI003416-4,SZAXPI003417-4,SZAXPI003419-6,...,CM.55_WGS,CM.57_WGS,CM.64_WGS,CM.65_WGS,CM.7_WGS,CM.70_WGS,CM.75_WGS,CM.80_WGS,CM.86_WGS,CM.90_WGS
0,Akkermansia_muciniphila,10.95862,0.01264,4.83142,11.62145,0.00418,0.00211,0.00181,0.00617,0.00123,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.0000
1,Escherichia_coli,0.34416,0.33701,0.81837,0.20090,1.97896,9.62665,0.56901,36.66533,0.00822,...,4.25066,3.96767,0.00000,2.51976,0.74695,6.56968,1.12418,5.40897,28.17015,0.0000
2,Faecalibacterium_prausnitzii,5.15137,0.81765,3.37249,2.29617,0.69536,0.00414,0.00141,0.13091,1.56926,...,3.10020,4.04407,4.95012,2.76194,2.60093,2.92987,6.17024,2.45086,1.37108,8.0989
3,Bacteroides_vulgatus,0.07968,0.19470,2.76035,5.97123,0.01097,1.18482,23.33269,10.99991,27.98496,...,0.00000,0.00000,0.00000,0.00000,0.39340,0.00593,0.00000,0.00000,0.00000,0.0000
4,Anaerostipes_hadrus,0.19203,0.65001,0.92489,0.23338,0.84590,0.78542,0.00000,0.01824,0.86415,...,0.00000,0.00000,0.70777,0.00000,0.00000,0.25758,0.00000,0.00000,0.00000,0.0000


<h1>ID checking</h1>

In [6]:
meta_samples = set(list(meta['sample_id']))
paths_samples = set(list(paths.columns[1:]))

In [7]:
print(len(meta_samples))
print(len(paths_samples))

print(meta_samples - paths_samples)
print(paths_samples - meta_samples)
print(meta_samples == paths_samples)

3752
3752
set()
set()
True


In [8]:
taxs_samples = set(list(taxs.columns[1:]))

In [9]:
print(taxs_samples == meta_samples)
print(paths_samples == meta_samples)

True
True


<h1>Abundance values</h1>

In [10]:
print(paths.sum())

Unnamed: 0    DTDPRHAMSYN-PWY:_dTDP-L-rhamnose_biosynthesis_...
CON-001                                                   100.0
CON-002                                                   100.0
CON-003                                                   100.0
CON-004                                                   100.0
                                    ...                        
H1M512855                                                 100.0
H2M414940                                                 100.0
nHM511825                                                 100.0
nHM612836                                                 100.0
nHM412840                                                 100.0
Length: 3753, dtype: object


In [11]:
print(taxs.sum())

Species           Akkermansia_muciniphilaEscherichia_coliFaecali...
SZAXPI003410-3                                             99.99609
SZAXPI003411-6                                             99.99433
SZAXPI003412-1                                             99.99651
SZAXPI003413-4                                             99.99526
                                        ...                        
CM.70_WGS                                                     100.0
CM.75_WGS                                                 100.00001
CM.80_WGS                                                  99.99997
CM.86_WGS                                                     100.0
CM.90_WGS                                                  99.99999
Length: 3753, dtype: object


<h1>Category of samples</h1>

In [12]:
category = meta['study_condition']
print(len(set(category)))
print(set(category))
category = Counter(meta['study_condition'])
print(category)

category = meta['study_condition']
category = [("healthy" if x == "control" else "diseased") for x in category]
print(Counter(category))

16
{'HF', 'hypertension', 'ME/CFS', 'T2D', 'pre-hypertension', 'CAD', 'STH', 'carcinoma_surgery_history', 'IGT', 'schizophrenia', 'cirrhosis', 'CRC', 'IBD', 'ACVD', 'adenoma', 'control'}
Counter({'control': 1651, 'CRC': 468, 'T2D': 389, 'IBD': 284, 'ACVD': 148, 'CAD': 138, 'cirrhosis': 109, 'hypertension': 99, 'schizophrenia': 76, 'STH': 70, 'IGT': 69, 'adenoma': 67, 'pre-hypertension': 56, 'ME/CFS': 50, 'carcinoma_surgery_history': 40, 'HF': 38})
Counter({'diseased': 2101, 'healthy': 1651})


<h1>Theoretical Introduction</h1>

1. Find out how MetaPhIAn and HumanN works
2. Describe data content (compositional data)

<h1>PCA analysis</h1>

1. CLR transformation
2. Find out which components have the most impact on data variance (pca.explained_variance_ratio_)
3. Plot PCA2
4. Perform this on taxonomy and pathways data
5. Filter by groups

In [13]:
def clr_transform(df):
    X = df.to_numpy(dtype=np.float64)
    X_log = np.log1p(X)
    X_clr = X_log - X_log.mean(axis=1, keepdims=True)
    return X_clr

<h1>Taxon-taxon interactions</h1>

1. Divide data into two groups: healthy, unhealthy
2. Use SparCC to determine significant correlations between taxons in groups
3. Build graph using output from SparCC
4. Check if we can see differences in graph structure between groups
5. Check if there are any node communities (using networkx build in functions)
6. Create ML classification models for each community to check their connection to inless status

<h1>Taxon-Function Interactions</h1>

1. 

<h1>Bibliography</h1>

1. https://github.com/dlegor/SparCC